# COMP9517 Traditional Handcrafted-Feature Pipeline


In [1]:
# Local VS Code setup is already complete. This cell verifies the selected kernel.
import sys
from pathlib import Path
print('Python:', sys.executable)
assert '.venv/bin/python' in sys.executable, 'Select the repository .venv kernel in VS Code'

Python: /Users/z5655976/Documents/COMP9517-Group-102/.venv/bin/python


In [2]:
REPO = Path('/Users/z5655976/Documents/COMP9517-Group-102')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATA = REPO / 'Team_Dataset'
OUTPUT = REPO / 'artifacts/traditional'
assert DATA.exists(), f'Dataset not found: {DATA}'
print(DATA.resolve(), OUTPUT.resolve())

/Users/z5655976/Documents/COMP9517-Group-102/Team_Dataset /Users/z5655976/Documents/COMP9517-Group-102/artifacts/traditional


## 1. Integrity audit and 500-image timing pilot

In [3]:
import sys

REPO = "/Users/z5655976/Documents/COMP9517-Group-102"

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(sys.path[0])

/Users/z5655976/Documents/COMP9517-Group-102


In [4]:
from traditional_cv.data import build_manifest
from traditional_cv.experiment import ExperimentRunner
manifest = build_manifest(DATA, expected_classes=500, hash_files=False)
runner = ExperimentRunner(manifest, OUTPUT)
display(manifest.frame.groupby('split').size())
display(runner.benchmark(sample_size=500))

split
test      5000
train    20000
val       5000
dtype: int64

Extracting features:   0%|          | 0/500 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/500 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/500 [00:00<?, ?it/s]

,feature,sample_images,seconds,estimated_30000_seconds
0,hsv,500,5.057130,303.427812
1,lbp,500,5.498230,329.893773
2,hog,500,4.893195,293.591705


## 2. Streamlined comprehensive experiments 
This runs exactly six controlled experiments: HSV, LBP, HOG and SIFT-256 with the same fast hinge-loss linear SVM; fused features with that SVM; and fused features with Random Forest. 

In [5]:
summary = runner.run_streamlined_experiments()
display(summary)

Stage 1/3: descriptor ablation

=== hsv + stochastic linear SVM ===

=== lbp + stochastic linear SVM ===


Extracting features:   0%|          | 0/20000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]


=== hog + stochastic linear SVM ===


Extracting features:   0%|          | 0/20000 [00:00<?, ?it/s]

...

Fitting SIFT vocabulary:   0%|          | 0/20000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/20000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]


Stage 2/3: feature-fusion ablation

Stage 3/3: classifier ablation


,features,classifier,macro_f1
5,hsv+lbp+hog+bovw256,random_forest,0.062424
4,hsv+lbp+hog+bovw256,stochastic_linear_svm,0.053145
2,hog,stochastic_linear_svm,0.019259
3,bovw256,stochastic_linear_svm,0.016322
0,hsv,stochastic_linear_svm,0.015407
1,lbp,stochastic_linear_svm,0.011579


## 3. Inspect and freeze validation selection


In [6]:
import json
selection = json.loads((OUTPUT / 'STREAMLINED_COMPLETE.json').read_text())
selection

{'experiments': 6,
 'selected_features': 'hsv+lbp+hog+bovw256',
 'selected_classifier': 'random_forest',
 'validation_macro_f1': 0.062423951356888746}

## 4. Final held-out test evaluation 

In [8]:
test_metrics = runner.evaluate_test_once()
test_metrics

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]

Extracting features:   0%|          | 0/5000 [00:00<?, ?it/s]

{'top1_accuracy': 0.0744,
 'top5_accuracy': 0.182,
 'overall_accuracy': 0.0744,
 'balanced_accuracy': 0.0744,
 'macro_precision': 0.0587469639420674,
 'macro_recall': 0.0744,
 'macro_f1': 0.06091650014577219,
 'inference_seconds': 8.690561541821808,
 'images_per_second': 575.336815226309}